# Phase 06B.03B — Deterministic temporal feature-bank build

Loads only manifest-pinned frozen encoders. Builds isolated train_fit, inner_dev and validation banks. Validation never contains relevance targets; smoke is never scientific completion.

In [ ]:
from pathlib import Path
import hashlib,json,sys,time
import numpy as np
import pandas as pd
import torch
from PIL import Image
ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/"src").is_dir()),None)
if ROOT is None: raise RuntimeError("Run inside RoadBuddy")
if str(ROOT/"src") not in sys.path: sys.path.insert(0,str(ROOT/"src"))
from roadbuddy_common import read_video_frames_at_indices,save_json
from phase06a_common import probe_video,sha256_file,sha256_json
from phase06b_common import validate_feature_bank_manifest,validate_temporal_input_manifest,validate_temporal_support_annotations
from traffic_temporal_grounding import indices_to_timestamps,uniform_candidate_indices,weak_targets_from_support_times
RUN_SCOPE="smoke" # smoke | full
LIMIT=8 if RUN_SCOPE=="smoke" else None
OUT=ROOT/"outputs/phase06b/feature_banks"/RUN_SCOPE; OUT.mkdir(parents=True,exist_ok=True)
INPUT=ROOT/"outputs/phase06b/temporal_inputs/temporal_input_manifest.json"; SUPPORT=ROOT/"data/phase06b/train_temporal_support.csv"
if RUN_SCOPE not in {"smoke","full"}: raise ValueError("Explicit smoke/full scope required")
if not INPUT.is_file():
 save_json(OUT/"PHASE06B_03B_STATUS.json",{"status":"awaiting_encoder_manifest","scope":RUN_SCOPE,"scientific_complete":False}); raise RuntimeError("Lock encoder manifest first")
inputs=validate_temporal_input_manifest(json.loads(INPUT.read_text()))


In [ ]:
from transformers import AutoImageProcessor,AutoModel,AutoTokenizer
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
visual_processor=AutoImageProcessor.from_pretrained(inputs["visual_encoder"],revision=inputs["visual_encoder_revision"],trust_remote_code=True)
visual_model=AutoModel.from_pretrained(inputs["visual_encoder"],revision=inputs["visual_encoder_revision"],trust_remote_code=True).to(device).eval()
question_tokenizer=AutoTokenizer.from_pretrained(inputs["question_encoder"],revision=inputs["question_encoder_revision"],trust_remote_code=True)
question_model=AutoModel.from_pretrained(inputs["question_encoder"],revision=inputs["question_encoder_revision"],trust_remote_code=True).to(device).eval()
for model in [visual_model,question_model]:
 for parameter in model.parameters(): parameter.requires_grad=False
def pooled(output):
 if hasattr(output,"pooler_output") and output.pooler_output is not None: return output.pooler_output
 if hasattr(output,"last_hidden_state"): return output.last_hidden_state.mean(dim=1)
 if isinstance(output,(tuple,list)): return output[0].mean(dim=1) if output[0].ndim==3 else output[0]
 raise ValueError("Encoder output needs pooler_output or last_hidden_state")
@torch.inference_mode()
def encode_frames(frames):
 batch=visual_processor(images=[Image.fromarray(frame) for frame in frames],return_tensors="pt"); batch={k:v.to(device) for k,v in batch.items()}
 if hasattr(visual_model,"get_image_features"): value=visual_model.get_image_features(**batch)
 else: value=pooled(visual_model(**batch))
 return torch.nn.functional.normalize(value.float(),dim=-1).cpu() if inputs["normalization"]=="l2" else value.float().cpu()
@torch.inference_mode()
def encode_question(text):
 batch=question_tokenizer(str(text),return_tensors="pt",truncation=True); batch={k:v.to(device) for k,v in batch.items()}
 if hasattr(question_model,"get_text_features"): value=question_model.get_text_features(**batch)
 else: value=pooled(question_model(**batch))
 value=value.float(); value=torch.nn.functional.normalize(value,dim=-1) if inputs["normalization"]=="l2" else value
 return value[0].cpu()


In [ ]:
train=pd.read_csv(ROOT/"data/splits/phase01/train.csv"); group_map=dict(zip(train.sample_id.astype(str),train.group_id.astype(str)))
if not SUPPORT.is_file() or sha256_file(SUPPORT)!=inputs["support_annotations_sha256"]: raise ValueError("Audited support file/hash mismatch")
support=validate_temporal_support_annotations(pd.read_csv(SUPPORT),expected_train_ids=train.sample_id,expected_group_by_id=group_map)
support_centers=support.assign(center=(support.support_start_sec+support.support_end_sec)/2).groupby("sample_id").center.apply(list).to_dict()
splits={"train_fit":ROOT/"data/splits/phase06a_inner/train_fit.csv","inner_dev":ROOT/"data/splits/phase06a_inner/inner_dev.csv","validation":ROOT/"data/splits/phase01/validation.csv"}
traffic_path=inputs.get("traffic_features_path"); traffic_store=None
if traffic_path:
 tp=ROOT/traffic_path
 if not tp.is_file() or sha256_file(tp)!=inputs.get("traffic_features_sha256"): raise ValueError("Traffic feature store/hash mismatch")
 traffic_store=torch.load(tp,map_location="cpu",weights_only=False)
reports={}


In [ ]:
for split,path in splits.items():
 frame=pd.read_csv(path); frame=frame.head(LIMIT) if LIMIT else frame
 if split!="validation": frame=frame[frame.sample_id.astype(str).isin(support_centers)].reset_index(drop=True)
 records=[]; video_hashes={}; question_hashes={}
 for _,row in frame.iterrows():
  metadata=probe_video(str(row.video_path)); valid_indices=uniform_candidate_indices(metadata["total_frames"],32); valid_times=indices_to_timestamps(valid_indices,metadata["fps"]); images=read_video_frames_at_indices(str(row.video_path),valid_indices); features=encode_frames(images)
  valid=len(valid_indices); pad=32-valid; frame_features=torch.cat([features,torch.zeros(pad,features.shape[1])]) if pad else features; valid_mask=torch.tensor([True]*valid+[False]*pad); indices=valid_indices+[-1]*pad; timestamps=valid_times+[0.0]*pad
  question_features=encode_question(row.question); rec={"sample_id":str(row.sample_id),"group_id":str(row.group_id),"split":split,"candidate_indices":indices,"candidate_timestamps_sec":timestamps,"normalized_timestamps":torch.tensor(timestamps)/max(metadata["duration_seconds"],1e-8),"frame_features":frame_features,"question_features":question_features,"valid_mask":valid_mask}
  if split!="validation": rec["relevance_targets"]=weak_targets_from_support_times(torch.tensor(timestamps[:valid]),torch.tensor(support_centers[str(row.sample_id)]),split_name="train").new_zeros(32); rec["relevance_targets"][:valid]=weak_targets_from_support_times(torch.tensor(timestamps[:valid]),torch.tensor(support_centers[str(row.sample_id)]),split_name="train")
  if traffic_store is not None:
   traffic=torch.as_tensor(traffic_store[str(row.sample_id)]);
   if traffic.shape[0]!=32: raise ValueError("Traffic features must align with all 32 candidates")
   rec["traffic_features"]=traffic
  video_hashes[str(row.sample_id)]=sha256_file(Path(row.video_path)); question_hashes[str(row.sample_id)]=hashlib.sha256(str(row.question).encode()).hexdigest(); records.append(rec)
 if not records: raise RuntimeError(f"No eligible records for {split}")
 split_out=OUT/split; split_out.mkdir(parents=True,exist_ok=True); bank=split_out/"feature_bank.pt"; torch.save(records,bank)
 manifest={"schema_version":2,"split":split,"rows":len(records),"membership_sha256":sha256_json(sorted(r["sample_id"] for r in records)),"candidate_count":32,"source_video_manifest_sha256":sha256_json(video_hashes),"question_manifest_sha256":sha256_json(question_hashes),"visual_encoder_sha256":inputs["visual_encoder_checkpoint_sha256"],"question_encoder_sha256":inputs["question_encoder_checkpoint_sha256"],"preprocessing_sha256":inputs["preprocessing_sha256"],"dtype":inputs["dtype"],"normalization":inputs["normalization"],"frame_feature_dim":int(records[0]["frame_features"].shape[1]),"question_feature_dim":int(records[0]["question_features"].shape[0]),"traffic_feature_dim":int(records[0].get("traffic_features",torch.empty(32,0)).shape[1]),"contains_relevance_targets":split!="validation","bank_sha256":sha256_file(bank),"scope":RUN_SCOPE}
 validate_feature_bank_manifest(manifest); save_json(split_out/"feature_bank_manifest.json",manifest); reports[split]=manifest
status="complete" if RUN_SCOPE=="full" else "smoke_complete"
save_json(OUT/"PHASE06B_03B_STATUS.json",{"status":status,"scope":RUN_SCOPE,"scientific_complete":RUN_SCOPE=="full","traffic_features":traffic_store is not None,"banks":{k:v["bank_sha256"] for k,v in reports.items()}})
reports
